# 1. Import Libraries

In [ ]:
import pandas as pd 
import re
from pathlib import Path

# 2. Merge Metadata Files

In [ ]:
METADATA_FOLDER = Path.cwd() / 'metadata_list'

metadata_df_list = []

for path_to_file in METADATA_FOLDER.glob('*.csv'):

    print(path_to_file)
    
    sub_metadata_df = pd.read_csv(path_to_file)

    metadata_df_list.append(sub_metadata_df)

metadata_df = pd.concat(metadata_df_list, ignore_index=True)

In [ ]:
for col in ['released_on', 'issued_on', 'adopted']:
    metadata_df[col] = pd.to_datetime(metadata_df[col], errors='coerce')

metadata_df = metadata_df.sort_values(
    'released_on', 
    ascending=False, 
    ignore_index=True
)

In [ ]:
metadata_df['download_status'].value_counts()

In [ ]:
metadata_df = metadata_df[metadata_df['download_status'] == 'downloaded']

In [ ]:
metadata_df['downloaded_filename'] = metadata_df['downloaded_files'].apply(lambda x: Path(str(x)).name)

metadata_df['downloaded_filename_key'] = metadata_df['downloaded_filename'].str.lower()

mask_duplicates = metadata_df['downloaded_filename_key'].duplicated(keep='first')

metadata_df = metadata_df[~mask_duplicates]

In [ ]:
metadata_df.to_csv('fcc_news_release_metadata.csv')

# 3. Extract the Text Body from News Releases

## 3.1. Regular Expressions for News Release Formats

In [ ]:
DATELINE_LINE_RE = re.compile(
    r'''(?ix)
    (?:
        ^\s*(?:[-–—]{1,2}|:)?\s*
        (?:
            (?:
                \(?\s*
                WASHINGTON
                (?:\s*,?\s*D\.?\s*C\.?)?
                \s*\)?
            )|
            (?:
                [A-Z][A-Z\s.'-]+
                ,?\s*CALIFORNIA
            )
        )
        \s*,?\s*\(?        
        [A-Z]{3,9}\.?,?\s+\d{1,2},\s+\d{4}
        \.?\s*\)?\s*
        (?:[-–—]{1,2}|:)?\s*
    )|
    (?:
        ^\s*\(?
        [A-Z]{3,9}\.?,?\s+\d{1,2},\s+\d{4}
        \.?\s*\)?\s*
        (?:[—–]|(?<!-)-(?!-))
        \s*(?=[A-Z])
    )
    '''
)

In [ ]:
DATELINE_PREFIX_RE = re.compile(
    r'''(?ix)
    ^\s*\(?\s*
    WASHINGTON
    (?:\s*,?\s*D\.?\s*C\.?)?
    \s*\)?\s*,?\s*
    (?:[-–—]{1,2}|:)?\s*
    '''
)

In [ ]:
DATE_LINE_RE = re.compile(
    r'''(?ix)
    ^
    (?:FOR\s+IMMEDIATE\s+RELEASE:?\s*)?
    (?:[A-Z]{3,9}\.?\s+\d{1,2},\s+\d{4})\b
    '''
)

In [ ]:
HEADER_RE = re.compile(
    r'''(?ix)
    ^(?:
        NEWS|
        ADVISORY|
        PUBLIC\s+NOTICE|
        Federal\s+Communications\s+Commission\s*$|
        445\s+12(?:th)?\s+Street|
        th$|
        Street,\s*S\.W\.|
        Washington,?\s*D\.?\s*C\.?\s*20554|
        This\s+is\s+an\s+unofficial\s+announcement|
        constitutes\s+official\s+action|
        See\s+MCI\s+v\.\s*FCC|
        News\s+Media\s+Information|
        Internet:|
        TTY:|
        Fax-On-Demand|
        ftp\.fcc\.gov|
        FOR\s+IMMEDIATE\s+RELEASE|
        NEWS\s+(?:MEDIA\s+)?CONTACTS?:?|
        MEDIA\s+CONTACT:?
    )
    '''
)

In [ ]:
CONTACT_RE = re.compile(
    r'''(?ix)
    (
        \bmedia\s+contact\b|
        \bnews\s+contact\b|
        ^contact\b|
        \bemail\b|
        \be-mail\b|
        @fcc\.gov|
        \(?202\)?[- .]?418|
        888[- .]?835
    )
    '''
)

In [ ]:
FOOTER_RE = re.compile(
    r'''(?ix)
    ^.*(?:\#\s*){3}\s*$|
    ^\s*[-–—]*\s*FCC\s*[-–—]*\s*$
    '''
)

In [ ]:
TITLE_SKIP_RE = re.compile(
    r'''(?ix)
    ^(?:
        [-–—]+|
        \(?[A-Z]{1,4}\s+DOCKET\s+NO\.?|
        Docket\s+No\.?|
        Re\s*:
    )
    '''
)

In [ ]:
BODY_START_RE = re.compile(
    r'''(?ix)
    (
        ^\s*\(?\s*
        WASHINGTON
        (?:\s*,?\s*D\.?\s*C\.?)?
        \s*\)?\s*,?\s*
        (?:[-–—]{1,2}|:)?\s*
    )?
    (?:
        After|As|At|Broadband|Cable|Chairman|Commissioner|
        Consumers|During|FCC|Federal|I|In|My|On|
        Preet|Thank|The|This|Today|Wireless|Without
    )\b
    '''
)

## 3.2. Define Helper Functions

In [ ]:
def normalize_lines(text):
    '''
    Convert raw text into clean non-empty lines.
    '''
    text = (
        text.replace('\ufeff', '')
            .replace('\r\n', '\n')
            .replace('\r', '\n')
            .replace('\xa0', ' ')
            .replace('�', '')
    )

    lines = []

    for line in text.splitlines():
        line = re.sub(r'\s+', ' ', line).strip()

        if line:
            lines.append(line)

    return lines

In [ ]:
def upper_ratio(line):
    '''
    Calculate how much of a line is uppercase.
    Useful for detecting title/header lines.
    '''
    letters = [char for char in line if char.isalpha()]

    if not letters:
        return 0

    return sum(char.isupper() for char in letters) / len(letters)

In [ ]:
def is_front_matter_line(line):
    '''
    Detect header/title/contact lines before the actual body.
    '''
    if HEADER_RE.search(line):
        return True

    if CONTACT_RE.search(line):
        return True

    if DATE_LINE_RE.search(line):
        return True

    if TITLE_SKIP_RE.search(line):
        return True

    # Many FCC titles are written in uppercase.
    if len(line) <= 180 and upper_ratio(line) >= 0.72 and not line.endswith(('.', '?', '!')):
        return True

    if len(line) <30:
        return True

    return False

In [ ]:
def looks_like_body_start(line):
    '''
    Detect a likely first body line.
    '''
    if is_front_matter_line(line):
        return False

    if len(line) < 30:
        return False

    # A normal sentence-like line.
    if re.search(r'[A-Z][a-z].*[\.;?!]$', line):
        return True

    if BODY_START_RE.match(line):
        return True

    return False

In [ ]:
def find_body_end(lines):
    '''
    Find the end of the substantive body before footer/contact blocks.
    '''
    for i, line in enumerate(lines):
        if i > len(lines) * 0.37 and FOOTER_RE.match(line):
            return i, 'footer_marker'

    # Some recent releases use trailing media contact blocks.
    for i, line in enumerate(lines):
        if i > len(lines) * 0.37 and re.match(r'(?i)Media Contact:', line):
            return i, 'trailing_media_contact'

        if i > len(lines) * 0.37 and re.match(r'(?i)Office of Media Relations:', line):
            return i, 'office_media_footer'

    return len(lines), 'no_footer_marker'

In [ ]:
def clean_join(lines):
    '''
    Join wrapped PDF/text lines into one clean text string.
    '''
    body = ' '.join(lines)

    # Remove spaces before punctuation.
    body = re.sub(r'\s+([,.;:?!])', r'\1', body)

    # Collapse repeated spaces.
    body = re.sub(r'\s+', ' ', body).strip()

    return body

In [ ]:
def extract_body(text):
    '''
    Extract the main body of one FCC news release.

    Parameters
    ----------
    text : str
        Raw text from one .txt file.

    Returns
    -------
    body_text : str
    extraction_method : str
    footer_method : str
    '''
    lines = normalize_lines(text)

    end_idx, footer_method = find_body_end(lines)
    
    lines = lines[:end_idx]

    # Method 1: Most news releases have a WASHINGTON dateline.
    for i, line in enumerate(lines):
        if DATELINE_LINE_RE.match(line):
            body_lines = lines[i:]

            body_lines[0] = DATELINE_LINE_RE.sub('', body_lines[0]).strip()
            body_lines = [line for line in body_lines if line]

            return clean_join(body_lines), 'dateline', footer_method

    # Method 2: If no dateline exists, skip front matter/title/contact lines.
    i = 0

    while i < min(60, len(lines)) and is_front_matter_line(lines[i]):
        i += 1

    for j in range(i, len(lines)):
        if looks_like_body_start(lines[j]):
            body_lines= lines[j:]

            body_lines[0] = DATELINE_PREFIX_RE.sub('', body_lines[0]).strip()
            body_lines = [line for line in body_lines if line]
            
            return clean_join(body_lines), 'fallback_body_like_after_front_matter', footer_method

    # Method 3: Last fallback.
    for j, line in enumerate(lines):
        if looks_like_body_start(line):
            return clean_join(lines[j:]), 'fallback_body_like_global', footer_method

    return clean_join(lines[i:]), 'fallback_after_front_matter', footer_method

In [ ]:
def build_body_dataframe_from_folder(text_folder):
    text_folder = Path(text_folder)

    records = []

    for file_path in sorted(text_folder.glob('*.txt')):
        try:
            raw_text = file_path.read_text(encoding='utf-8-sig')
        except UnicodeDecodeError:
            raw_text = file_path.read_text(encoding='cp1252', errors='replace')

        body_text, extraction_method, footer_method = extract_body(raw_text)

        records.append({
            'filename': file_path.name,
            'body_text': body_text,
            'body_word_count': len(body_text.split()),
            'extraction_method': extraction_method,
            'footer_method': footer_method
        })

    return pd.DataFrame(records)

## 3.3. Collect the Text

In [ ]:
TEXT_FOLDER = Path('fcc_news_release_documents')

body_df = build_body_dataframe_from_folder(TEXT_FOLDER)

body_df.head()

In [ ]:
print('Empty body texts:', (body_df['body_word_count'] == 0).sum())

In [ ]:
body_df.to_csv(
    'fcc_news_release_body_texts.csv',
    index=False,
    encoding='utf-8-sig'
)

In [ ]:
body_df['filename_key'] = body_df['filename'].str.lower()

metadata_df = metadata_df.merge(
    body_df,
    left_on='downloaded_filename_key',
    right_on='filename_key',
    how='left'
)

metadata_df = metadata_df.drop(columns=[
    'selected_txt_count', 'download_status', 'error_message',
    'downloaded_files', 'downloaded_filename', 'downloaded_filename_key', 'filename_key'
    ]
)

In [ ]:
remove_file_list = [
    '210624_Commissioner_Simington_Participates_On_TIA_Broadband_Panel.txt',
    # FCC-provided original .txt file is encoding-corrupted
    # https://www.fcc.gov/document/commissioner-simington-participates-tia-broadband-panel
    
    '111130_Broadband_Adoption_Presentation_to_FCC_Open_Meeting.txt',
    # Empty .txt file
    # >>> https://www.fcc.gov/document/broadband-adoption-presentation-fcc-open-meeting
    
    '110516_List_Of_Input_Channels_For_Tv_Translators_Stations_As_Of_May_16,_2011.txt'
    # Tabular reference list, not a substantive news release
    # >>> https://www.fcc.gov/document/list-input-channels-tv-translators-stations-may-16-2011
]

for url in metadata_df.loc[
    metadata_df['filename'].isin(remove_file_list),
    'webpage_url'
]:
    print(url)

In [ ]:
metadata_df = metadata_df[
    ~metadata_df['filename'].isin(remove_file_list)
]

metadata_df

In [ ]:
print('Number of documents:', len(metadata_df))

print('\n', metadata_df['extraction_method'].value_counts())

print('\n', metadata_df['footer_method'].value_counts())

LENGTH_THRESHOLD =50
text_is_short = metadata_df['body_word_count'] < LENGTH_THRESHOLD
print(f'\n Short text (less than {LENGTH_THRESHOLD} words): {text_is_short.sum()}')

## 3.4. Inspect the Text

In [ ]:
inspect_filter = metadata_df['footer_method'] == 'no_footer_marker'

In [ ]:
metadata_df[inspect_filter][['filename', 'body_text', 'extraction_method', 'footer_method']]

In [ ]:
for filename in metadata_df[inspect_filter]['filename']:
    print(filename)

In [ ]:
for idx in metadata_df[inspect_filter].index:

    filename = metadata_df.loc[idx, 'filename']
    print(f'<File name {idx}: {filename}> \n')

    file_path = TEXT_FOLDER / filename
    text = file_path.read_text(encoding='utf-8-sig')
    print(f'[Original text] >>> {normalize_lines(text)} \n')
       
    print(f'[Text body] >>> {extract_body(text)} \n')
    print()

In [ ]:
comparison_output_path = Path('full_text_vs_cleaned_body.txt')

comparison_blocks = []

for idx in metadata_df.index:

    filename = metadata_df.loc[idx, 'filename']

    file_path = TEXT_FOLDER / filename
    text = file_path.read_text(encoding='utf-8-sig')

    original_lines = normalize_lines(text)
    body_text, extraction_method, footer_method = extract_body(text)

    comparison_block = f'''
============================================================
File index: {idx}
File name: {filename}
Extraction method: {extraction_method}
Footer method: {footer_method}
============================================================

[Original text]

{'\n'.join(original_lines)}

------------------------------------------------------------

[Cleaned text body]

{body_text}

'''

    comparison_blocks.append(comparison_block)


comparison_output_path.write_text(
    '\n'.join(comparison_blocks),
    encoding='utf-8-sig'
)

In [ ]:
filename = '100120_FCC_PROPOSES_TOUGHER_RESTRICTIONS_ON_ROBOCALLS.txt'

def inspect_body_extraction_by_line(filename, text_folder=TEXT_FOLDER):
    
    file_path = text_folder / filename
    text = file_path.read_text(encoding='utf-8-sig')
    for i, line in enumerate(normalize_lines(text)):
        print(f'Line {i}: {line}')
        print('>>> DATELINE LINE RE match:', DATELINE_LINE_RE.match(line))
        print('>>> is front matter line:', is_front_matter_line(line))
        print('>>> looks like body start:', looks_like_body_start(line))
        print()
    
    print(extract_body(text))

inspect_body_extraction_by_line(filename)